## Install

In [ ]:
!python -m spacy download es_core_news_sm
!pip install spacy[transformers]
!pip install mglearn
!python -m spacy download es_dep_news_trf
!pip install mglearn
!pip install pyLDAvis
#!pip install pyLDAvis.gensim

# Librerias

In [ ]:
import pandas as pd
import spacy
from google.colab import drive
from nltk import word_tokenize
from nltk import SnowballStemmer
import nltk
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import numpy as np
import mglearn
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
from gensim import corpora, models
from pprint import pprint
import pyLDAvis
import pyLDAvis.gensim_models
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)
from sklearn.manifold import TSNE
from bokeh.plotting import figure, output_file, show
from bokeh.models import Label
from bokeh.io import output_notebook
import matplotlib.colors as mcolors
from bokeh.core.enums import MarkerType
import seaborn as sns
sns.set_style("darkgrid")
%matplotlib inline
nltk.download('punkt')

In [ ]:
drive.mount('/content/drive/')

# Importamos Videos, Comentarios o Respuestas Comentarios

In [ ]:
# ======================== OJO Modificar para cada serie ========================

pathVD='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Series/\
Narcos/Comentarios/RESULTS-PAISCOMENTARIOS-Lematizado2022-09-13 17_51_55.913996.csv'

# ======================== OJO Modificar para cada serie ========================

dfVD= pd.read_csv(pathVD)
dfVD.info()

# Lematizacion (Si no se ha hecho, algo demorada)

## Lematizacion Profunda

In [ ]:
nlp = spacy.load("es_dep_news_trf")
nlp

In [ ]:
def lematizarprofundo(texto):
  doc_spacy = nlp(texto)
  tokens_lema= " ".join([token.lemma_ for token in doc_spacy])
  return tokens_lema

In [ ]:
dfVD['TotalTextoVideoModificadoLematizadoProfundo']=dfVD['TotalTextoVideoModificado'].apply(lematizarprofundo)

In [ ]:
dfVD['TotalTextoVideoModificadoLematizadoProfundo'][0]

## Lematizacion Debil

In [ ]:
es_nlp = spacy.load('es_core_news_sm')
es_nlp

In [ ]:
def lematizar(texto):
  doc_spacy = es_nlp(texto)
  tokens_lema= " ".join([token.lemma_ for token in doc_spacy])
  return tokens_lema

In [ ]:
dfVD['TotalTextoVideoModificadoLematizado']=dfVD['TotalTextoVideoModificado'].apply(lematizar)

In [ ]:
dfVD['TotalTextoVideoModificadoLematizado'][0]

In [ ]:
dfVD['TotalTextoVideoModificado'][0]

# LDA

## LDA con Gensim

### Preprocesamiento de datos

In [ ]:
columna='ComentarioPrincipalModificadoLematizadoProfundo' # 'TotalTextoVideoModificadoLematizadoProfundo', 'ComentarioPrincipalModificadoLematizadoProfundo' , 'ComentarioRespuestaModificadoLematizadoProfundo'

In [ ]:
#dfVD[columna] = dfVD[columna].apply(str.replace, args=('ano', 'año'))

#### Preparamos datos

In [ ]:
def tokenizar(texto):
  return word_tokenize(texto)

In [ ]:
dfVD['Tokenizado']=dfVD[columna].apply(tokenizar)
dfVD['Tokenizado']

In [ ]:
Tokenizado=list(dfVD['Tokenizado'])
print(Tokenizado)

#### Crear el diccionario (bolsa de palabras y TF-IDF) para el modelado de temas

In [ ]:
# Diccionario o vector de caracteristicas
id2word = corpora.Dictionary(Tokenizado)
count = 0
for k, v in id2word.iteritems():
  print(k, v)
  count += 1
  if count > 10:
      break

In [ ]:
# Cantidad de vocabulario
len(id2word)

#### Representacion bolsa de palabras (corpus)

In [ ]:
# Crear Corpus
texts = Tokenizado
# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]
print(corpus)

In [ ]:
len(corpus)

In [ ]:
# View
print(corpus[:1])

In [ ]:
len(corpus[0])

In [ ]:
corpus[0]

In [ ]:
len(corpus[:1])

In [ ]:
# Termino-Frecuencia  para primer video
[[(id2word[id], freq) for id, freq in cp] for cp in corpus[:1]]

#### Representacion TF-IDF (corpus)

In [ ]:
tfidf = models.TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]
corpus_tfidf

In [ ]:
len(corpus_tfidf[0])

In [ ]:
print(corpus_tfidf[0])

j=1
for doc in corpus_tfidf:
  print("video numero ",j)
  pprint(doc)
  j=j+1

#### Bigramas

In [ ]:
# Build the bigram and trigram models
bigram = gensim.models.Phrases(texts, min_count=1, delimiter=b' ') # higher threshold fewer phrases.

In [ ]:
# Faster way to get a sentence clubbed as a trigram/bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)

In [ ]:
len(texts[0])

In [ ]:
texts[0]

In [ ]:
len(bigram_mod[texts[0]])

In [ ]:
# See bigram example
print(bigram_mod[texts[0]])

In [ ]:
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

In [ ]:
# Form Bigrams
data_words_bigrams = make_bigrams(texts)
data_words_bigrams

#### Crear el diccionario (bigramas) para el modelado de temas

In [ ]:
# Create Dictionary 
id2word_bigrams = corpora.Dictionary(data_words_bigrams) 
len(id2word_bigrams)

#### Representacion bigramas (Corpus)

In [ ]:
# Create Corpus 
texts_bigrams = data_words_bigrams
# Term Document Frequency 
corpus_bigrams = [id2word_bigrams.doc2bow(text) for text in texts_bigrams]  
# View 
print(corpus_bigrams[:1])

In [ ]:
len(corpus_bigrams[0])

In [ ]:
len(corpus_bigrams)

### Numero optimo de temas para LDA (Solo si no se ha ejecutado)

In [ ]:
serie='Narcos' # 'EscobarPatrondelMal', 'Dignity', 'Narcos', 'Los80,MasqueUnaModa', 'TresCaines'
carpeta='Comentarios' # 'Videos', 'Comentarios', 'RespuestasComentarios'
datos='comentarios' # 'respuestas en comentarios', 'comentarios', 'videos con ruido'
tv='Narcos' #'EPM', 'Narcos', 'Dignity', 'Los 80', 'Tres Caines'

In [ ]:
def compute_coherence_values(dictionary, corpus, texts, limit, start=2, step=6):
  """
  Compute c_v coherence for various number of topics


  Parameters:
  ----------
  dictionary : Gensim dictionary
  corpus : Gensim corpus
  texts : List of input texts
  limit : Max num of topics

  Returns:
  -------
  model_list : List of LDA topic models
  coherence_values : Coherence values corresponding to the LDA model with respective number of topics
  """
  coherence_values = []
  model_list = []
  for num_topics in range(start, limit, step):
      model = gensim.models.ldamodel.LdaModel(corpus=corpus, num_topics=num_topics, id2word=dictionary,iterations=50,random_state=100,passes=20)
      model_list.append(model)
      coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
      coherence_values.append(coherencemodel.get_coherence())

  return model_list, coherence_values

In [ ]:
# Can take a long time to run.
model_list1, coherence_values1 = compute_coherence_values(dictionary=id2word, corpus=corpus, texts=Tokenizado, start=2, limit=40, step=2)

In [ ]:
# Show graph
limit=40; start=2; step=2;
x = range(start, limit, step)
plt.title("Puntaje coherencia vs Numero de topicos.\n Para {} en serie {} con representacion bolsa de palabras".format(datos,tv))
plt.plot(x, coherence_values1)
plt.xlabel("Numero de topicos")
plt.ylabel("Puntaje Coherencia")
plt.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/NumeroTopicosRepresentacionBolsaPalabras"+carpeta+serie+".png",bbox_inches='tight')
plt.show()

In [ ]:
# Print the coherence scores
for m, cv in zip(x, coherence_values1):
    print("Numero de topicos =", m, " Con un valor de coherencia de", round(cv, 4))

In [ ]:
# Select the model and print the topics
optimal_model1 = model_list1[3]
model_topics1 = optimal_model1.show_topics(formatted=False)
print(optimal_model1.print_topics(num_words=10))

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(optimal_model1, corpus, id2word)
vis

In [ ]:
model_list2, coherence_values2 = compute_coherence_values(dictionary=id2word, corpus=corpus_tfidf, texts=Tokenizado, start=2, limit=40, step=2)

In [ ]:
# Show graph
limit=40; start=2; step=2;
x = range(start, limit, step)
plt.title("Puntaje coherencia vs Numero de topicos.\n Para {} en serie {} con representacion TF-IDF".format(datos,tv))
plt.plot(x, coherence_values2)
plt.xlabel("Numero de topicos")
plt.ylabel("Puntaje Coherencia")
plt.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/NumeroTopicosRepresentacionTF-IDF"+carpeta+serie+".png",bbox_inches='tight')
plt.show()

In [ ]:
# Print the coherence scores
for m, cv in zip(x, coherence_values2):
    print("Numero de topicos =", m, " Con un valor de coherencia de", round(cv, 4))

In [ ]:
# Select the model and print the topics
optimal_model2 = model_list2[2]
model_topics2 = optimal_model2.show_topics(formatted=False)
print(optimal_model2.print_topics(num_words=10))

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(optimal_model2, corpus_tfidf, id2word)
vis

In [ ]:
model_list3, coherence_values3 = compute_coherence_values(dictionary=id2word_bigrams, corpus=corpus_bigrams, texts=data_words_bigrams, start=2, limit=40, step=2)

In [ ]:
# Show graph
limit=40; start=2; step=2;
x = range(start, limit, step)
plt.title("Puntaje coherencia vs Numero de topicos.\n Para {} en serie {} con representacion de bigramas".format(datos,tv))
plt.plot(x, coherence_values3)
plt.xlabel("Numero de topicos")
plt.ylabel("Puntaje Coherencia")
plt.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/NumeroTopicosRepresentacionBigramas"+carpeta+serie+".png",bbox_inches='tight')
plt.show()

In [ ]:
# Print the coherence scores
for m, cv in zip(x, coherence_values3):
    print("Numero de topicos =", m, " Con un valor de coherencia de", round(cv, 4))

In [ ]:
# Select the model and print the topics
optimal_model3 = model_list3[1]
model_topics3 = optimal_model3.show_topics(formatted=False)
print(optimal_model3.print_topics(num_words=10))

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(optimal_model3, corpus_bigrams, id2word_bigrams)
vis

### Construir el topic model OJO ==> Asignar numero de topicos

In [ ]:
NumTopics1=4
NumTopics2=4
NumTopics3=4
lda_model1 = gensim.models.ldamodel.LdaModel(corpus=corpus,id2word=id2word,num_topics=NumTopics1,random_state=100,passes=20,alpha='auto',iterations=50)
lda_model2 = gensim.models.ldamodel.LdaModel(corpus=corpus_tfidf,id2word=id2word,num_topics=NumTopics2,random_state=100,passes=20,alpha='auto',iterations=50)
lda_model3 = gensim.models.ldamodel.LdaModel(corpus=corpus_bigrams,id2word=id2word_bigrams,num_topics=NumTopics3,random_state=100,passes=20,alpha='auto',iterations=50)

In [ ]:
lda_model1.save("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/Modelo"+carpeta+serie+"lda_model1(BolsaPalabras)")
lda_model2.save("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/Modelo"+carpeta+serie+"lda_model2(TF_IDF)")
lda_model3.save("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/Modelo"+carpeta+serie+"lda_model3(Bigramas)")

In [ ]:
#lda_model1 =  models.LdaModel.load('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/Comentarios/Modelo'+datos+serie+'lda_model1')

### Ver los topicos en el modelo LDA 

In [ ]:
for idx, topic in lda_model1.print_topics(num_topics=-1,num_words=12):
  print('Topic: {} \nWords: {}'.format(idx, topic))

In [ ]:
for idx, topic in lda_model2.print_topics(num_topics=-1,num_words=12):
  print('Topic: {} \nWords: {}'.format(idx, topic))

In [ ]:
for idx, topic in lda_model3.print_topics(num_topics=-1,num_words=12):
  print('Topic: {} \nWords: {}'.format(idx, topic))

In [ ]:
Vocabulario1=len(id2word)
Vocabulario1

In [ ]:
Vocabulario2=len(id2word_bigrams)
Vocabulario2

In [ ]:
lda_model1.show_topics(num_topics=-1,num_words=Vocabulario1,formatted=False)

In [ ]:
lda_model2.show_topics(num_topics=-1,num_words=Vocabulario1,formatted=False)

In [ ]:
lda_model3.show_topics(num_topics=-1,num_words=Vocabulario2,formatted=False)

In [ ]:
len(lda_model1.show_topics(num_topics=-1,num_words=Vocabulario1,formatted=False))

In [ ]:
len(lda_model2.show_topics(num_topics=-1,num_words=Vocabulario1,formatted=False))

In [ ]:
len(lda_model3.show_topics(num_topics=-1,num_words=Vocabulario2,formatted=False))

In [ ]:
x,y=lda_model1.show_topics(num_topics=-1,num_words=Vocabulario1,formatted=False)[0]
len(y)

In [ ]:
x,y=lda_model2.show_topics(num_topics=-1,num_words=Vocabulario1,formatted=False)[0]
len(y)

In [ ]:
x,y=lda_model3.show_topics(num_topics=-1,num_words=Vocabulario2,formatted=False)[0]
len(y)

In [ ]:
lda_model1.get_topics().shape

In [ ]:
lda_model2.get_topics().shape

In [ ]:
lda_model3.get_topics().shape

In [ ]:
lda_model1.get_topics()

In [ ]:
lda_model2.get_topics()

In [ ]:
lda_model3.get_topics()

In [ ]:
caracteristicas1=[]
for k, v in id2word.iteritems():
  caracteristicas1.append(v)

In [ ]:
caracteristicas2=[]
for k, v in id2word_bigrams.iteritems():
  caracteristicas2.append(v)

In [ ]:
len(caracteristicas1)

In [ ]:
len(caracteristicas2)

In [ ]:
sorting1 = np.argsort(lda_model1.get_topics(), axis=1)[:, ::-1]
feature_names1 = np.array(caracteristicas1)

In [ ]:
sorting2 = np.argsort(lda_model2.get_topics(), axis=1)[:, ::-1]
feature_names2 = np.array(caracteristicas1)

In [ ]:
sorting3 = np.argsort(lda_model3.get_topics(), axis=1)[:, ::-1]
feature_names3 = np.array(caracteristicas2)

In [ ]:
sorting1

In [ ]:
sorting2

In [ ]:
sorting3

In [ ]:
sorting1.shape

In [ ]:
sorting2.shape

In [ ]:
sorting3.shape

In [ ]:
documentos1=[]
for i in lda_model1.__getitem__(bow=corpus):
  documentos1.append(i)

In [ ]:
documentos2=[]
for i in lda_model2.__getitem__(bow=corpus_tfidf):
  documentos2.append(i)

In [ ]:
documentos3=[]
for i in lda_model3.__getitem__(bow=corpus_bigrams):
  documentos3.append(i)

In [ ]:
len(documentos1)

In [ ]:
len(documentos2)

In [ ]:
len(documentos3)

In [ ]:
matrix1=[]
for i in range(NumTopics1):
  puntaje=[]
  for j in range(len(documentos1)):
    documentos1[j]
    for k in range(len(documentos1[j])):
      x,y=documentos1[j][k]
      if (x==i):
        puntaje.append(y)
        break
  matrix1.append(puntaje)

In [ ]:
matrix2=[]
for i in range(NumTopics2):
  puntaje=[]
  for j in range(len(documentos2)):
    documentos2[j]
    for k in range(len(documentos2[j])):
      x,y=documentos2[j][k]
      if (x==i):
        puntaje.append(y)
        break
  matrix2.append(puntaje)

In [ ]:
matrix3=[]
for i in range(NumTopics3):
  puntaje=[]
  for j in range(len(documentos3)):
    documentos3[j]
    for k in range(len(documentos3[j])):
      x,y=documentos3[j][k]
      if (x==i):
        puntaje.append(y)
        break
  matrix3.append(puntaje)

In [ ]:
len(matrix1)

In [ ]:
len(matrix2)

In [ ]:
len(matrix3)

In [ ]:
suma1=[]
for i in range(len(matrix1)):
  suma1.append(sum(matrix1[i]))

In [ ]:
suma2=[]
for i in range(len(matrix2)):
  suma2.append(sum(matrix2[i]))

In [ ]:
suma3=[]
for i in range(len(matrix3)):
  suma3.append(sum(matrix3[i]))

In [ ]:
len(suma1)

In [ ]:
len(suma2)

In [ ]:
len(suma3)

### Posibles titulos de topicos y palabras clave -> OJO (Organizar numero de graficas)

In [ ]:
serie='Narcos' # 'EscobarPatrondelMal', 'Dignity', 'Narcos', 'Los80,MasqueUnaModa', 'TresCaines'
carpeta='Comentarios' # 'Videos', 'Comentarios', 'RespuestasComentarios'
datos='comentarios' # 'respuestas en comentarios', 'comentarios', 'videos con ruido'
tv='Narcos' #'EPM', 'Narcos', 'Dignity', 'Los 80', 'Tres Caines'

#### Bolsa de palabras

In [ ]:
def plot_top_words(lda_model, feature_names, n_top_words, title,w,v,representacion):
    fig, axes = plt.subplots(w, v, figsize=(27, 12), sharex=True) # ancho, largo
    axes = axes.flatten()
    for topic_idx, topic in enumerate(lda_model.get_topics()):
        top_features_ind = topic.argsort()[: -n_top_words - 1 : -1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f"Tema {topic_idx}", fontdict={"fontsize": 20})
        ax.invert_yaxis()
        ax.tick_params(axis="both", which="major", labelsize=12)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=20, y=1.0)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=2, hspace=0.3)
    plt.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/PalabrasClavesTemasCon"+representacion+carpeta+serie+".png",bbox_inches='tight')
    plt.show()
    
    
x=plot_top_words(lda_model1,feature_names1,30,'Palabras claves en temas ({}) para {}.\n Serie {} con representacion bolsa de palabras'.format(NumTopics1,datos,tv),1,4,'BolsaPalabras')

In [ ]:
topics = np.arange(0,NumTopics1)
mglearn.tools.print_topics(topics=topics, feature_names=feature_names1,sorting=sorting1, topics_per_chunk=int(NumTopics1/2), n_words=15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12 , 4))
fig.suptitle('Pesos promedio de cada tema ({}) para {}.\n Serie {} con representacion bolsa de palabras'.format(NumTopics1,datos,tv), fontsize=16, y=1.08)

topic_names1 = ["Tema {:>2}: ".format(i) + " ".join(words) for i, words in enumerate(feature_names1[sorting1[:, :2]])]

# two column bar chart:
for col in [0, 1]:
  start = col * int(NumTopics1/2)
  end = (col + 1) * int(NumTopics1/2)
  ax[col].barh(np.arange(int(NumTopics1/2)), (np.array(suma1)/len(documentos1))[start:end])
  ax[col].set_yticks(np.arange(NumTopics1/2))
  ax[col].set_yticklabels(topic_names1[start:end], ha="left", va="top")
  ax[col].invert_yaxis()
  ax[col].set_xlim(0, 0.4)
  yax = ax[col].get_yaxis()
  yax.set_tick_params(pad=200)
plt.subplots_adjust(right=0.2)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/ModeladodeTemasConBolsadePalabras"+carpeta+serie+".png",bbox_inches='tight')

In [ ]:
topic_names1

#### Tf-IDF

In [ ]:
plot_top_words(lda_model2,feature_names2,30,'Palabras claves en temas ({}) para {}.\n Serie {} con representacion TF-IDF'.format(NumTopics2,datos,tv),1,4,'TF-IDF')

In [ ]:
topics = np.arange(0,NumTopics2)
mglearn.tools.print_topics(topics=topics, feature_names=feature_names2,sorting=sorting2, topics_per_chunk=int(NumTopics2/2), n_words=15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12 , 4))
fig.suptitle('Pesos promedio de cada tema ({}) para {}.\n Serie {} con representacion TF-IDF'.format(NumTopics2,datos,tv), fontsize=16, y=1.08)

topic_names2 = ["Tema {:>2}: ".format(i) + " ".join(words) for i, words in enumerate(feature_names2[sorting2[:, :2]])]

# two column bar chart:
for col in [0, 1]:
  start = col * int(NumTopics2/2)
  end = (col + 1) * int(NumTopics2/2)
  ax[col].barh(np.arange(int(NumTopics2/2)), (np.array(suma2)/len(documentos2))[start:end])
  ax[col].set_yticks(np.arange(NumTopics2/2))
  ax[col].set_yticklabels(topic_names2[start:end], ha="left", va="top")
  ax[col].invert_yaxis()
  ax[col].set_xlim(0, 0.5)
  yax = ax[col].get_yaxis()
  yax.set_tick_params(pad=200)
plt.subplots_adjust(right=0.2)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/ModeladodeTemasConTF-IDF"+carpeta+serie+".png",bbox_inches='tight')

In [ ]:
topic_names2

#### Bigramas

In [ ]:
plot_top_words(lda_model3,feature_names3,30,'Palabras claves en temas ({}) para {}.\n Serie {} con representacion de bigramas'.format(NumTopics3,datos,tv),1,4,'Bigramas')

In [ ]:
topics = np.arange(0,NumTopics3)
mglearn.tools.print_topics(topics=topics, feature_names=feature_names3,sorting=sorting3, topics_per_chunk=int(NumTopics3/2), n_words=15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12 , 4))
fig.suptitle('Pesos promedio de cada tema ({}) para {}.\n Serie {} con representacion de bigramas'.format(NumTopics3,datos,tv), fontsize=16, y=1.08)

topic_names3 = ["Tema {:>2}: ".format(i) + " ".join(words) for i, words in enumerate(feature_names3[sorting3[:, :2]])]

# two column bar chart:
for col in [0, 1]:
  start = col * int(NumTopics3/2)
  end = (col + 1) * int(NumTopics3/2)
  ax[col].barh(np.arange(int(NumTopics3/2)), (np.array(suma3)/len(documentos3))[start:end])
  ax[col].set_yticks(np.arange(NumTopics3/2))
  ax[col].set_yticklabels(topic_names3[start:end], ha="left", va="top")
  ax[col].invert_yaxis()
  ax[col].set_xlim(0, 0.4)
  yax = ax[col].get_yaxis()
  yax.set_tick_params(pad=200)
plt.subplots_adjust(right=0.2)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/ModeladodeTemasConBigramas"+carpeta+serie+".png",bbox_inches='tight')

In [ ]:
topic_names3

###Perplejidad del modelo y coherencia

In [ ]:
# Compute Perplexity
print('\nPerplexity en representacion bolsa de palabras: ', lda_model1.log_perplexity(corpus))  # a measure of how good the model is. lower the better.

# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model1, texts=Tokenizado, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score en representacion bolsa de palabras: ', coherence_lda)

In [ ]:
# Compute Perplexity
print('\nPerplexity en representacion TF-IDF: ', lda_model2.log_perplexity(corpus_tfidf))  # a measure of how good the model is. lower the better.

# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model2, texts=Tokenizado, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score en representacion TF-IDF: ', coherence_lda)

In [ ]:
# Compute Perplexity
print('\nPerplexity en representacion Bigramas: ', lda_model3.log_perplexity(corpus_bigrams))  # a measure of how good the model is. lower the better.

# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model3, texts=data_words_bigrams, dictionary=id2word_bigrams, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score en representacion Bigramas: ', coherence_lda)

###Visualizando topicos

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model1, corpus, id2word)
vis

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model2, corpus_tfidf, id2word)
vis

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model3, corpus_bigrams, id2word_bigrams)
vis

### Encontrar el tema dominante en cada documento

#### Funciones

In [ ]:
def format_topics_sentences_bolsapalabras(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get main topic in each document
    for i, row in enumerate(ldamodel[corpus]):
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        # Get the Dominant topic, Perc Contribution and Keywords for each document
        for j, (topic_num, prop_topic) in enumerate(row):
            if j == 0:  # => dominant topic
                wp = ldamodel.show_topic(topic_num)
                topic_keywords = ", ".join([word for word, prop in wp])
                sent_topics_df = sent_topics_df.append(pd.Series([int(topic_num), round(prop_topic,4), topic_keywords]), ignore_index=True)
            else:
                break
    sent_topics_df.columns = ['TemaDominante(BolsaPalabras)', 'ContribucionPorcentaje(BolsaPalabras)', 'PalabrasClave(BolsaPalabras)']

    # Add original text to the end of the output
    contents = texts
    sent_topics_df = pd.concat([sent_topics_df, contents], axis=1)
    return(sent_topics_df)

def format_topics_sentences_tf_idf(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get main topic in each document
    for i, row in enumerate(ldamodel[corpus]):
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        # Get the Dominant topic, Perc Contribution and Keywords for each document
        for j, (topic_num, prop_topic) in enumerate(row):
            if j == 0:  # => dominant topic
                wp = ldamodel.show_topic(topic_num)
                topic_keywords = ", ".join([word for word, prop in wp])
                sent_topics_df = sent_topics_df.append(pd.Series([int(topic_num), round(prop_topic,4), topic_keywords]), ignore_index=True)
            else:
                break
    sent_topics_df.columns = ['TemaDominante(tf_idf)', 'ContribucionPorcentaje(tf_idf)', 'PalabrasClave(tf_idf)']

    # Add original text to the end of the output
    contents = texts
    sent_topics_df = pd.concat([sent_topics_df, contents], axis=1)
    return(sent_topics_df)


def format_topics_sentences_bigramas(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get main topic in each document
    for i, row in enumerate(ldamodel[corpus]):
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        # Get the Dominant topic, Perc Contribution and Keywords for each document
        for j, (topic_num, prop_topic) in enumerate(row):
            if j == 0:  # => dominant topic
                wp = ldamodel.show_topic(topic_num)
                topic_keywords = ", ".join([word for word, prop in wp])
                sent_topics_df = sent_topics_df.append(pd.Series([int(topic_num), round(prop_topic,4), topic_keywords]), ignore_index=True)
            else:
                break
    sent_topics_df.columns = ['TemaDominante(Bigramas)', 'ContribucionPorcentaje(Bigramas)', 'PalabrasClave(Bigramas)']

    # Add original text to the end of the output
    contents = texts
    sent_topics_df = pd.concat([sent_topics_df, contents], axis=1)
    return(sent_topics_df)

In [ ]:
df_topic_sents_keywords1 = format_topics_sentences_bolsapalabras(ldamodel=lda_model1, corpus=corpus, texts=dfVD)
df_topic_sents_keywords2 = format_topics_sentences_tf_idf(ldamodel=lda_model2, corpus=corpus_tfidf, texts=df_topic_sents_keywords1)
df_topic_sents_keywords3 = format_topics_sentences_bigramas(ldamodel=lda_model3, corpus=corpus_bigrams, texts=df_topic_sents_keywords2)

In [ ]:
pd.unique(df_topic_sents_keywords3['TemaDominante(BolsaPalabras)'])

In [ ]:
NewnumTopics1=len(pd.unique(df_topic_sents_keywords3['TemaDominante(BolsaPalabras)']))
NewnumTopics1

In [ ]:
pd.unique(df_topic_sents_keywords3['TemaDominante(tf_idf)'])

In [ ]:
NewnumTopics2=len(pd.unique(df_topic_sents_keywords3['TemaDominante(tf_idf)']))
NewnumTopics2

In [ ]:
pd.unique(df_topic_sents_keywords3['TemaDominante(Bigramas)'])

In [ ]:
NewnumTopics3=len(pd.unique(df_topic_sents_keywords3['TemaDominante(Bigramas)']))
NewnumTopics3

In [ ]:
# Show
df_topic_sents_keywords3.head(10)

#### Exportando modelos de temas

In [ ]:
serie='Narcos'

In [ ]:
ruta='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Series/\
'+serie+'/'+carpeta

#ruta='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Series/\
#'+serie

In [ ]:
nombre='ComentariosTemas' # 'VideosRuidoconTemas' , 'ComentariosTemas', 'RespuestasTemas'
df_topic_sents_keywords3.to_csv(ruta+'/'+nombre+'.csv', index=False)

In [ ]:
df_topic_sents_keywords3.info()

In [ ]:
df_topic_sents_keywords3[df_topic_sents_keywords3['TemaDominante(BolsaPalabras)']==0.0]

In [ ]:
len(df_topic_sents_keywords3[df_topic_sents_keywords3['TemaDominante(BolsaPalabras)']==0.0])

#### Bolsa de palabras

##### Documento mas representativo para cada tema

In [ ]:
# Group top 5 sentences under each topic
sent_topics_sorteddf_mallet = pd.DataFrame()

sent_topics_outdf_grpd = df_topic_sents_keywords3.groupby('TemaDominante(BolsaPalabras)')

for i, grp in sent_topics_outdf_grpd:
    sent_topics_sorteddf_mallet = pd.concat([sent_topics_sorteddf_mallet, 
                                             grp.sort_values(['ContribucionPorcentaje(BolsaPalabras)'], ascending=[0]).head(1)], 
                                            axis=0)

# Reset Index    
sent_topics_sorteddf_mallet.reset_index(drop=True, inplace=True)


# Show
sent_topics_sorteddf_mallet

##### Distribución de temas en los documentos 

In [ ]:
# Number of Documents for Each Topic
topic_counts = df_topic_sents_keywords3['TemaDominante(BolsaPalabras)'].value_counts()

# Percentage of Documents for Each Topic
topic_contribution = round(topic_counts/topic_counts.sum(), 4)

# Topic Number and Keywords
topic_num_keywords = sent_topics_sorteddf_mallet[['TemaDominante(BolsaPalabras)', 'PalabrasClave(BolsaPalabras)']]


In [ ]:
x=pd.DataFrame(topic_counts).reset_index()
x.columns = ['TemaDominante(BolsaPalabras)', 'ConteoDocumentos(BolsaPalabras)']
x

In [ ]:
y=pd.DataFrame(topic_contribution).reset_index()
y.columns = ['TemaDominante(BolsaPalabras)', 'PorcentajeDocumentos(BolsaPalabras)']
y

In [ ]:
w=pd.merge(x,y,left_on='TemaDominante(BolsaPalabras)',right_on='TemaDominante(BolsaPalabras)')
w

In [ ]:
z=topic_num_keywords
z

In [ ]:
df_dominant_topics=pd.merge(w,z,left_on='TemaDominante(BolsaPalabras)',right_on='TemaDominante(BolsaPalabras)')
df_dominant_topics

In [ ]:
bolsa=df_dominant_topics[['TemaDominante(BolsaPalabras)','ConteoDocumentos(BolsaPalabras)']].sort_values(by='ConteoDocumentos(BolsaPalabras)',ascending=False, ignore_index=True)
bolsa

In [ ]:
bolsa['TemaDominante(BolsaPalabras)']=bolsa['TemaDominante(BolsaPalabras)'].astype('int')
bolsa

In [ ]:
topic_names1

In [ ]:
dic_topic_names1={}
for i in range(len(topic_names1)):
  dic_topic_names1[i]=topic_names1[i]
dic_topic_names1

In [ ]:
list(dic_topic_names1.keys())

In [ ]:
list(dic_topic_names1.values())

In [ ]:
diccionario_Final1={'TemaDominante(BolsaPalabras)':list(dic_topic_names1.keys()),'Nombre Tema':list(dic_topic_names1.values())}
diccionario_Final1

In [ ]:
temas1=pd.DataFrame(diccionario_Final1)
temas1

In [ ]:
bolsa=pd.merge(bolsa,temas1,left_on='TemaDominante(BolsaPalabras)',right_on='TemaDominante(BolsaPalabras)')
bolsa

In [ ]:
bolsa.columns = ['Tema Dominante','Videos','Tema']

In [ ]:
bolsa

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title('Distribucion de temas ({}) para {}.\n Serie {} con representacion de bolsa de palabras'.format(NewnumTopics1,datos,tv))
grafica1=sns.barplot(x='Videos', y='Tema',data=bolsa, palette="crest")
plt.show()

In [ ]:
grafica1.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/DistribuciondeTemasconBolsaPalabras"+carpeta+serie+".png",bbox_inches='tight')

#### TF-IDF

##### Documento mas representativo para cada tema

In [ ]:
# Group top 5 sentences under each topic
sent_topics_sorteddf_mallet = pd.DataFrame()

sent_topics_outdf_grpd = df_topic_sents_keywords3.groupby('TemaDominante(tf_idf)')

for i, grp in sent_topics_outdf_grpd:
    sent_topics_sorteddf_mallet = pd.concat([sent_topics_sorteddf_mallet, 
                                             grp.sort_values(['ContribucionPorcentaje(tf_idf)'], ascending=[0]).head(1)], 
                                            axis=0)

# Reset Index    
sent_topics_sorteddf_mallet.reset_index(drop=True, inplace=True)


# Show
sent_topics_sorteddf_mallet

##### Distribución de temas en los documentos 

In [ ]:
# Number of Documents for Each Topic
topic_counts = df_topic_sents_keywords3['TemaDominante(tf_idf)'].value_counts()

# Percentage of Documents for Each Topic
topic_contribution = round(topic_counts/topic_counts.sum(), 4)

# Topic Number and Keywords
topic_num_keywords = sent_topics_sorteddf_mallet[['TemaDominante(tf_idf)', 'PalabrasClave(tf_idf)']]

In [ ]:
x=pd.DataFrame(topic_counts).reset_index()
x.columns = ['TemaDominante(tf_idf)', 'ConteoDocumentos(tf_idf)']
x

In [ ]:
y=pd.DataFrame(topic_contribution).reset_index()
y.columns = ['TemaDominante(tf_idf)', 'PorcentajeDocumentos(tf_idf)']
y

In [ ]:
w=pd.merge(x,y,left_on='TemaDominante(tf_idf)',right_on='TemaDominante(tf_idf)')
w

In [ ]:
z=topic_num_keywords
z

In [ ]:
df_dominant_topics=pd.merge(w,z,left_on='TemaDominante(tf_idf)',right_on='TemaDominante(tf_idf)')
df_dominant_topics

In [ ]:
tf=df_dominant_topics[['TemaDominante(tf_idf)','ConteoDocumentos(tf_idf)']].sort_values(by='ConteoDocumentos(tf_idf)',ascending=False, ignore_index=True)
tf

In [ ]:
tf['TemaDominante(tf_idf)']=tf['TemaDominante(tf_idf)'].astype('int')
tf

In [ ]:
topic_names2

In [ ]:
dic_topic_names2={}
for i in range(len(topic_names2)):
  dic_topic_names2[i]=topic_names2[i]
dic_topic_names2

In [ ]:
list(dic_topic_names2.keys())

In [ ]:
list(dic_topic_names2.values())

In [ ]:
diccionario_Final2={'TemaDominante(tf_idf)':list(dic_topic_names2.keys()),'Nombre Tema':list(dic_topic_names2.values())}
diccionario_Final2

In [ ]:
temas2=pd.DataFrame(diccionario_Final2)
temas2

In [ ]:
tf=pd.merge(tf,temas2,left_on='TemaDominante(tf_idf)',right_on='TemaDominante(tf_idf)')
tf

In [ ]:
tf.columns = ['Tema Dominante','Videos','Tema']

In [ ]:
tf

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title('Distribucion de temas ({}) para {}.\n Serie {} con representacion TF-IDF'.format(NewnumTopics2,datos,tv))
grafica1=sns.barplot(x='Videos', y='Tema',data=tf, palette="crest")
plt.show()

In [ ]:
grafica1.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/DistribuciondeTemasconTF_IDF"+carpeta+serie+".png",bbox_inches='tight')

#### Bigramas

##### Documento mas representativo para cada tema

In [ ]:
# Group top 5 sentences under each topic
sent_topics_sorteddf_mallet = pd.DataFrame()

sent_topics_outdf_grpd = df_topic_sents_keywords3.groupby('TemaDominante(Bigramas)')

for i, grp in sent_topics_outdf_grpd:
    sent_topics_sorteddf_mallet = pd.concat([sent_topics_sorteddf_mallet, 
                                             grp.sort_values(['ContribucionPorcentaje(Bigramas)'], ascending=[0]).head(1)], 
                                            axis=0)

# Reset Index    
sent_topics_sorteddf_mallet.reset_index(drop=True, inplace=True)


# Show
sent_topics_sorteddf_mallet

##### Distribución de temas en los documentos

In [ ]:
# Number of Documents for Each Topic
topic_counts = df_topic_sents_keywords3['TemaDominante(Bigramas)'].value_counts()

# Percentage of Documents for Each Topic
topic_contribution = round(topic_counts/topic_counts.sum(), 4)

# Topic Number and Keywords
topic_num_keywords = sent_topics_sorteddf_mallet[['TemaDominante(Bigramas)', 'PalabrasClave(Bigramas)']]

In [ ]:
x=pd.DataFrame(topic_counts).reset_index()
x.columns = ['TemaDominante(Bigramas)', 'ConteoDocumentos(Bigramas)']
x

In [ ]:
y=pd.DataFrame(topic_contribution).reset_index()
y.columns = ['TemaDominante(Bigramas)', 'PorcentajeDocumentos(Bigramas)']
y

In [ ]:
w=pd.merge(x,y,left_on='TemaDominante(Bigramas)',right_on='TemaDominante(Bigramas)')
w

In [ ]:
z=topic_num_keywords
z

In [ ]:
df_dominant_topics=pd.merge(w,z,left_on='TemaDominante(Bigramas)',right_on='TemaDominante(Bigramas)')
df_dominant_topics

In [ ]:
Bigramas=df_dominant_topics[['TemaDominante(Bigramas)','ConteoDocumentos(Bigramas)']].sort_values(by='ConteoDocumentos(Bigramas)',ascending=False, ignore_index=True)
Bigramas

In [ ]:
Bigramas['TemaDominante(Bigramas)']=Bigramas['TemaDominante(Bigramas)'].astype('int')

In [ ]:
topic_names3

In [ ]:
dic_topic_names3={}
for i in range(len(topic_names3)):
  dic_topic_names3[i]=topic_names3[i]
dic_topic_names3

In [ ]:
list(dic_topic_names3.keys())

In [ ]:
list(dic_topic_names3.values())

In [ ]:
diccionario_Final3={'TemaDominante(Bigramas)':list(dic_topic_names3.keys()),'Nombre Tema':list(dic_topic_names3.values())}
diccionario_Final3

In [ ]:
temas3=pd.DataFrame(diccionario_Final3)
temas3

In [ ]:
Bigramas=pd.merge(Bigramas,temas3,left_on='TemaDominante(Bigramas)',right_on='TemaDominante(Bigramas)')
Bigramas

In [ ]:
Bigramas.columns = ['Tema Dominante','Videos','Tema']

In [ ]:
Bigramas

In [ ]:
fig, axes = plt.subplots(figsize=(12,10))
plt.title('Distribucion de temas ({}) para {}.\n Serie {} con representacion de Bigramas'.format(NewnumTopics3,datos,tv))
grafica1=sns.barplot(x='Videos', y='Tema',data=Bigramas, palette="crest")
plt.show()

In [ ]:
grafica1.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/YoutubeReport/"+carpeta+"/ModelToping/"+serie+"/DistribuciondeTemasconBigramas"+carpeta+serie+".png",bbox_inches='tight')

### Clustering Chart

In [ ]:
MaxTopic=4

#### No se ejecuta si el numero de topicos es menor igual a 10

In [ ]:
from random import randint
mapa = []
n = MaxTopic 
for i in range(n):
    mapa.append('#%06X' % randint(0, 0xFFFFFF))

In [ ]:
mapa

In [ ]:
dibujo=[]
x=True
while(x==True):
  for i, marker in enumerate(MarkerType):
    dibujo.append(marker)
    if (len(dibujo)==n):
      x=False
      break

In [ ]:
len(dibujo)

In [ ]:
marcaAlterna={}
for i in range(len(dibujo)):
  marcaAlterna[i]=dibujo[i]

#### Diccionarios de temas

In [ ]:
diclabel1={}
j=0
for i in range(NumTopics1):
  diclabel1[i]='Tema '+str(j)
  j=j+1

In [ ]:
diclabel2={}
j=0
for i in range(NumTopics2):
  diclabel2[i]='Tema '+str(j)
  j=j+1

In [ ]:
diclabel3={}
j=0
for i in range(NumTopics3):
  diclabel3[i]='Tema '+str(j)
  j=j+1

#### Marca (30) y Colores (10)

In [ ]:
for i, marker in enumerate(MarkerType):
  print(marker)

In [ ]:
dibujo=['asterisk','circle','diamond','star', 'triangle'\
        ,'square','plus','inverted_triangle','hex','circle_x'\
        ,'circle_y','diamond_dot','cross','triangle_dot','circle'\
        ,'circle','diamond','star', 'triangle','square'\
        ,'plus','inverted_triangle','hex','circle_x','circle_y'\
        ,'diamond_dot','square','triangle_dot','circle','circle']
len(dibujo)
marca={}
for i in range(len(dibujo)):
  marca[i]=dibujo[i]

In [ ]:
len(dibujo)

In [ ]:
mycolors = np.array([color for name, color in mcolors.TABLEAU_COLORS.items()])
len(mycolors)

#### Reduccion de dimensionalidad

###### Matrices

In [ ]:
a1 = np.zeros((len(lda_model1.get_document_topics(bow=corpus)), NumTopics1))
a2 = np.zeros((len(lda_model2.get_document_topics(bow=corpus_tfidf)), NumTopics2))
a3 = np.zeros((len(lda_model3.get_document_topics(bow=corpus_bigrams)), NumTopics3))

In [ ]:
for i in range(len(documentos1)):
  for j in range(len(documentos1[i])):
    x,y=documentos1[i][j]
    a1[i][x]=y

In [ ]:
for i in range(len(documentos2)):
  for j in range(len(documentos2[i])):
    x,y=documentos2[i][j]
    a2[i][x]=y

In [ ]:
for i in range(len(documentos3)):
  for j in range(len(documentos3[i])):
    x,y=documentos3[i][j]
    a3[i][x]=y

In [ ]:
a1

In [ ]:
a2

In [ ]:
a3

In [ ]:
a1.shape

In [ ]:
a2.shape

In [ ]:
a3.shape

In [ ]:
# Dominant topic number in each doc
topic_num1 = np.argmax(a1, axis=1)
topic_num2 = np.argmax(a2, axis=1)
topic_num3 = np.argmax(a3, axis=1)

# tSNE Dimension Reduction
tsne_model = TSNE(n_components=2, random_state=100, init='pca',n_iter=2000)

##### Representacion bolsa de palabras

In [ ]:
tsne_lda1 = tsne_model.fit_transform(a1)

In [ ]:
topic_num1.shape

In [ ]:
topic_num1

In [ ]:
np.unique(topic_num1)

In [ ]:
NewnumTopics1=len(np.unique(topic_num1))
NewnumTopics1

In [ ]:
x1=tsne_lda1[:,0]
len(x1)
y1=tsne_lda1[:,1]
len(y1)

###### Dataframe de representacion bolsa de palabras (Graficas)

In [ ]:
df1 = pd.DataFrame(dict(x=tsne_lda1[:,0], y=tsne_lda1[:,1], label=topic_num1))
groups1 = df1.groupby('label')
df1

##### Representacion TF-IDF

In [ ]:
tsne_lda2 = tsne_model.fit_transform(a2)

In [ ]:
topic_num2.shape

In [ ]:
topic_num2

In [ ]:
np.unique(topic_num2)

In [ ]:
NewnumTopics2=len(np.unique(topic_num2))
NewnumTopics2

In [ ]:
x2=tsne_lda2[:,0]
len(x2)
y2=tsne_lda2[:,1]
len(y2)

###### Dataframe de representacion TF-IDF (Graficas)

In [ ]:
df2 = pd.DataFrame(dict(x=tsne_lda2[:,0], y=tsne_lda2[:,1], label=topic_num2))
groups2 = df2.groupby('label')
df2

##### Representacion Bigramas

In [ ]:
tsne_lda3 = tsne_model.fit_transform(a3)

In [ ]:
topic_num3.shape

In [ ]:
topic_num3

In [ ]:
np.unique(topic_num3)

In [ ]:
NewnumTopics3=len(np.unique(topic_num3))
NewnumTopics3

In [ ]:
x3=tsne_lda3[:,0]
len(x3)
y3=tsne_lda3[:,1]
len(y3)

###### Dataframe de representacion Bigramas (Graficas)

In [ ]:
df3 = pd.DataFrame(dict(x=tsne_lda3[:,0], y=tsne_lda3[:,1], label=topic_num3))
groups3 = df3.groupby('label')
df3

### Graficas

#### Bolsa de palabras

In [ ]:
# Plot the Topic Clusters using Bokeh
output_notebook()
plot = figure(title="Cluster con t-SNE  de {} temas para {}.\n Serie {} con representacion de bolsa de palabras".format(NewnumTopics1,datos,tv),plot_width=700, plot_height=700)
for name, group in groups1:
  plot.scatter(x=group.x, y=group.y, color=mycolors[name], legend_label=diclabel1[name], alpha=0.7, marker=marca[name], size=8, line_color="black", line_width=0.5)
plot.legend.label_text_font_size = '12pt'
show(plot)

#### TF-IDF

In [ ]:
# Plot the Topic Clusters using Bokeh
output_notebook()
plot = figure(title="Cluster con t-SNE  de {} temas para {}.\n Serie {} con representacion TF-IDF".format(NewnumTopics2,datos,tv),plot_width=700, plot_height=700)
for name, group in groups2:
  plot.scatter(x=group.x, y=group.y, color=mycolors[name], legend_label=diclabel2[name], alpha=0.7, marker=marca[name], size=8, line_color="black", line_width=0.5)
plot.legend.label_text_font_size = '12pt'
show(plot)

#### Bigramas

In [ ]:
# Plot the Topic Clusters using Bokeh
output_notebook()
plot = figure(title="Cluster con t-SNE  de {} temas para {}.\n Serie {} con representacion de bigramas".format(NewnumTopics3,datos,tv),plot_width=700, plot_height=750)
for name, group in groups3:
  plot.scatter(x=group.x, y=group.y, color=mycolors[name], legend_label=diclabel3[name], alpha=0.7, marker=marca[name], size=8, line_color="black", line_width=0.5)
plot.legend.label_text_font_size = '12pt'
show(plot)